# Task 3 - Model Comparison (Kaggle/Colab)
Load outputs from SARIMA/LSTM/GRU and build plots + tables.

In [ ]:
# Load model outputs
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Search current working output first, then attached datasets for exported CSVs.
SEARCH_ROOTS = [Path("/kaggle/working/outputs"), Path("outputs"), Path("/kaggle/input")]
SEARCH_ROOTS = [root for root in SEARCH_ROOTS if root.exists()]
print("Search roots:")
for root in SEARCH_ROOTS:
    print(" -", root.resolve())

def find_file(filename: str) -> Path | None:
    for root in SEARCH_ROOTS:
        matches = list(root.rglob(filename)) if root.is_dir() else []
        if matches:
            return matches[0]
    return None

def load_metrics(filename: str, model_name: str) -> pd.DataFrame | None:
    path = find_file(filename)
    if path is None:
        print(f"Missing metrics file for {model_name}: {filename}")
        return None
    df = pd.read_csv(path)
    df["model"] = model_name
    df["source_file"] = str(path)
    return df

metrics_frames = [
    load_metrics("metrics_sarima.csv", "SARIMA"),
    load_metrics("metrics_lstm.csv", "LSTM"),
    load_metrics("metrics_gru.csv", "GRU"),
]
metrics_frames = [frame for frame in metrics_frames if frame is not None]
if not metrics_frames:
    raise FileNotFoundError("No metrics CSV files were found in working output or attached datasets.")
metrics = pd.concat(metrics_frames, ignore_index=True)
metrics

In [ ]:
# Metrics tables by square
metrics_table = metrics.pivot_table(index=["square_id", "model"], values=["MAE", "MAPE", "RMSE"]).reset_index()
metrics_table

In [ ]:
# Forecast overlay plots (9 total)
def plot_overlay(csv_name: str, title: str):
    csv_path = find_file(csv_name)
    if csv_path is None:
        print(f"Skipping missing forecast file: {csv_name}")
        return None
    df = pd.read_csv(csv_path)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(pd.to_datetime(df["time_interval"], utc=True), df["y_true"], label="Observed", linewidth=1.8)
    ax.plot(pd.to_datetime(df["time_interval"], utc=True), df["y_pred"], label="Predicted", linewidth=1.6, alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel("Internet traffic")
    ax.legend()
    fig.tight_layout()
    return fig

for model_name in ["sarima", "lstm", "gru"]:
    for square_id in metrics["square_id"].unique():
        csv_name = f"{model_name}_{int(square_id)}.csv"
        plot_overlay(csv_name, f"{model_name.upper()} - Square {int(square_id)} (Dec 16-22)")